
# Policy & Simulation Suite — **Operational Reference**

This notebook lets you **experiment safely** with adaptive policies:
- Continuous temperature scheduling
- Lateness setpoint control
- EMA rate-limit feedback under burst/stall
- Monte Carlo over λ distributions
- Yield vs bandwidth tradeoff curves


## 1. Core Scheduling Imports

In [ ]:

import numpy as np, math, random
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta

class RateController:
    def __init__(self, target_rps=10.0, ema_alpha=0.03, eta=0.05, g_min=0.5, g_max=3.0):
        self.target = target_rps; self.alpha = ema_alpha; self.eta = eta
        self.g = 1.0; self.g_min = g_min; self.g_max = g_max; self.rate_ema = target_rps
    def update(self, instantaneous_rps, queue_lateness=None, lateness_setpoint=None, eta_l=0.02):
        self.rate_ema = self.alpha * instantaneous_rps + (1 - self.alpha) * self.rate_ema
        e_r = (self.rate_ema / self.target) - 1.0
        d = self.eta * e_r
        if queue_lateness is not None and lateness_setpoint is not None:
            d += eta_l * (queue_lateness - lateness_setpoint)
        self.g *= math.exp(d); self.g = max(self.g_min, min(self.g_max, self.g))
        return self.g

def simulate_continuous_scheduler(sim_minutes=30, n_posts=50, lambda_mean=0.15,
                                  burst_prob=0.05, stall_prob=0.03, target_rps=10.0,
                                  lateness_setpoint=0.15, seed=42):
    rng = np.random.default_rng(seed)
    lambdas = rng.lognormal(mean=np.log(lambda_mean), sigma=0.5, size=n_posts)
    temps = np.ones(n_posts)
    next_scrape = np.zeros(n_posts)
    ctrl = RateController(target_rps=target_rps)

    burst_timer = 0; stall_timer = 0
    rate_hist, ema_hist, gain_hist, info_hist = [], [], [], []
    info_yield, total_requests = 0.0, 0

    SECONDS = int(sim_minutes*60)
    for t in range(SECONDS):
        temps *= np.exp(-lambdas/3600.0)  # decay
        ready = np.where(t >= next_scrape)[0]
        if burst_timer==0 and stall_timer==0:
            if rng.random() < burst_prob: burst_timer = rng.integers(3, 10)
            elif rng.random() < stall_prob: stall_timer = rng.integers(30, 240)
        if stall_timer>0:
            R_t = 0; stall_timer -= 1
        elif burst_timer>0:
            R_t = min(len(ready), int(rng.integers(20, 40))); burst_timer -= 1
        else:
            R_t = min(len(ready), int(target_rps + rng.normal(0,2)))

        if R_t>0 and len(ready)>0:
            chosen = rng.choice(ready, size=min(len(ready), R_t), replace=False)
            total_requests += len(chosen)
            info_gain = temps[chosen] * rng.uniform(0.5, 1.0, size=len(chosen))
            info_yield += float(info_gain.sum())
            for i in chosen:
                k=1.5; d_t = k/(temps[i]+1e-3); d_t = np.clip(d_t, 0.5, 24)*ctrl.g
                next_scrape[i] = t + d_t*3600
                temps[i] = max(temps[i], 0.01)

        ctrl.update(R_t)
        rate_hist.append(R_t); ema_hist.append(ctrl.rate_ema); gain_hist.append(ctrl.g); info_hist.append(info_yield)

    return {"rate_hist": rate_hist, "ema_hist": ema_hist, "gain_hist": gain_hist, "info_hist": info_hist,
            "total_requests": total_requests, "total_info": info_yield}

def plot_sim(result, target_rps=10.0):
    t_min = np.arange(len(result["rate_hist"])) / 60.0
    fig, ax1 = plt.subplots(figsize=(11,5))
    ax1.plot(t_min, result["rate_hist"], color='grey', alpha=0.35, label='Instant req/s')
    ax1.plot(t_min, result["ema_hist"], color='blue', label='EMA req/s')
    ax1.axhline(target_rps, color='red', linestyle='--', label='Target')
    ax1.set_xlabel('Minutes'); ax1.set_ylabel('Req/s'); ax1.legend(loc='upper left')
    ax2 = ax1.twinx()
    ax2.plot(t_min, result["gain_hist"], color='green', label='Gain g', alpha=0.8)
    ax2.plot(t_min, result["info_hist"], color='purple', label='Cumulative ΔInfo', alpha=0.7)
    ax2.set_ylabel('Gain / Cumulative ΔInfo'); ax2.legend(loc='upper right')
    plt.title('Throughput Stability & Yield')
    plt.tight_layout(); plt.show()


## 2. Experiments

In [ ]:

res = simulate_continuous_scheduler(sim_minutes=15, n_posts=50, target_rps=10.0)
plot_sim(res)
print("Total requests:", res["total_requests"], "Total info:", round(res["total_info"], 2))
